In [1]:
# Pre process PM2.5 data from climate model, this will potentially be
# climate model specific, a few examples are outlined below

In [2]:
import os
import json
import xarray as xr
from utils.utils import get_scenario_config, fix_months

In [3]:
def load_file_list(DIR, filename):
    file_path = os.path.join(DIR, filename)
    with open(file_path, "r") as f:
        data = json.load(f)
    return data["files"]

In [4]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

FILE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/file_paths/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/processed_pm25/"

In [5]:
# === Find variable name ===

# Load first ensemble as a test
file_list = load_file_list(FILE_DIR, f"file_list_{scenario}_1.json")
# Open first file in the list
ds = xr.open_dataset(file_list[0])

# Print list of variables in data set - CHOOSE THE ONE NEEDED FOR PM2.5
print(ds.data_vars)

Data variables:
    zlon_bnds     (zlon, nbnd) float64 16B ...
    gw            (lat) float64 2kB ...
    hyam          (lev) float64 560B ...
    hybm          (lev) float64 560B ...
    P0            float64 8B ...
    hyai          (ilev) float64 568B ...
    hybi          (ilev) float64 568B ...
    ndbase        float64 8B ...
    nsbase        float64 8B ...
    nbdate        float64 8B ...
    nbsec         float64 8B ...
    mdt           float64 8B ...
    date          (time) float64 5kB ...
    datesec       (time) float64 5kB ...
    time_bnds     (time, nbnd) object 10kB ...
    date_written  (time) object 5kB ...
    time_written  (time) object 5kB ...
    ndcur         (time) float64 5kB ...
    nscur         (time) float64 5kB ...
    co2vmr        (time) float64 5kB ...
    ch4vmr        (time) float64 5kB ...
    n2ovmr        (time) float64 5kB ...
    f11vmr        (time) float64 5kB ...
    f12vmr        (time) float64 5kB ...
    sol_tsi       (time) float64 5kB 

In [14]:
# === Find units ===

# Select variable needed from above
var = "PM25"
da = ds[var]

# Print and save which units are being used
units = da.units
print(units)

kg/m3


In [15]:
# === Select surface level ===

# Print the coordinates of the data array
# You can choose which height level to look at from this
print(da.coords)

# === EXAMPLE ===
# For CESM2 lev: 992.6 is the surface and the final datapoint so
# we would select it like this:
da = da.isel(lev=-1)

Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * lev      (lev) float64 560B 5.96e-06 9.827e-06 1.62e-05 ... 976.3 992.6
  * time     (time) object 5kB 2015-02-01 00:00:00 ... 2065-01-01 00:00:00


In [19]:
# this will change depending on the model - need to see UKESM1 data to write this better
def select_surface(da):
    return da.isel(lev=-1)


def kgm3_to_µgm3(data):
    # kg/m3 -> µg/m3 (multiply by 1e9)
    return data * 1e9


def load_pm25(ds, var):
    da = ds[var]
    da = select_surface(da)
    units = da.units

    # Add conversions as needed
    if units == "kg/m3":
        da = kgm3_to_µgm3(da)
    elif units == "µg/m3":
        print("No conversion needed")
    else:
        print("Unit conversion not recognised")

    return da

In [20]:
# === MAIN LOOP ===

for ens_num in ensemble_members:
    print(f"Processing {scenario}, ensemble {ens_num:02d}")
    file_list = load_file_list(FILE_DIR, f"file_list_{scenario}_{ens_num}.json")
    datasets = []

    for f in file_list:
        if not os.path.exists(f):
            raise ValueError(f"Missing: {f}")

        print(f"Reading {os.path.basename(f)}")
        # var SPECIFIED ABOVE
        da = load_pm25(xr.open_dataset(f), var)
        datasets.append(da)

    # Combine files if multiple
    combined_ds = xr.concat(datasets,
                            dim="time") if len(datasets) > 1 else datasets[0]

Processing SSP245_G6, ensemble 01
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.001.cam.h0.PM25.201501-206412.nc
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.001.cam.h0.PM25.206501-210012.nc
Processing SSP245_G6, ensemble 02
Reading b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.002.cam.h0.PM25.201501-206412.nc


KeyboardInterrupt: 

In [33]:
combined_ds.time[-1].dt.month

<xarray.DataArray 'month' ()> Size: 8B
array(1)
Coordinates:
    lev      float64 8B 992.6
    time     object 8B 2101-01-01 00:00:00
Attributes:
    long_name:  time
    bounds:     time_bnds